# E07 02 - LangGraph (Resolution)

Ahora convertimos la chain de E07 01 en un grafo.

Flujo:

```txt
START -> greet_node -> format_node -> END
```

Conceptos nuevos:

- `State`: memoria compartida del flujo;
- `Node`: funcion que lee/escribe estado;
- `Edge`: conexion entre nodos;
- `compile`: valida el grafo;
- `invoke`: ejecuta el grafo.


## Antes de tocar codigo: que estamos construyendo

Este notebook esta pensado para que puedas entenderlo aunque lo abras sin ver la clase.

Tema: **LangGraph**.

La regla didactica es:

1. primero explicamos el concepto;
2. despues mostramos el codigo minimo;
3. despues conectamos ese codigo con el paso anterior;
4. finalmente ejecutamos y leemos el resultado.

Cuando veas una funcion, preguntate:

- que recibe;
- que devuelve;
- que parte del flujo representa;
- si es logica de negocio, orquestacion o instrumentacion.


In [ ]:
# Esta celda instala las dependencias del notebook.
# En Google Colab cada notebook arranca con un entorno limpio, por eso instalamos al inicio.
# En local, si ya instalaste estos paquetes, pip simplemente confirmara que existen.
!pip install -q langchain langchain-openai langgraph

print('Dependencias instaladas: langchain langchain-openai langgraph')


## Credenciales

Necesitamos OpenAI para que el nodo `greet_node` llame al LLM.


In [ ]:
import os
from getpass import getpass

# Nunca escribimos una API key real dentro del notebook.
# getpass permite pegar la key en ejecucion sin que quede visible en la salida.
# os.environ guarda la key solo para esta sesion de Python.
os.environ['OPENAI_API_KEY'] = getpass('OpenAI API Key: ')

print('OpenAI configurado para esta sesion.')


## Imports y State

`TypedDict` define la forma del estado. No ejecuta nada, pero documenta el contrato.

`StateGraph` es el builder del grafo.

`START` y `END` son nodos especiales: no tienen codigo propio, marcan entrada y salida.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

# El modelo se usa dentro del nodo greet_node.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

class GreetState(TypedDict):
    # Entrada original del flujo.
    name: str

    # Campo que escribira greet_node.
    message: str

    # Campo que escribira format_node.
    formatted: str

print('State y modelo listos.')


## Nodos

Un nodo de LangGraph es una funcion Python.

Regla importante:

```python
def node(state) -> dict:
    return {'campo_modificado': valor}
```

No hace falta devolver todo el estado. LangGraph fusiona el dict parcial con el estado existente.


In [ ]:
def greet_node(state: GreetState) -> dict:
    # Leemos solo lo que necesitamos del estado.
    name = state['name']

    # Construimos el prompt local de este nodo.
    prompt = f'Saluda a {name} en una oracion breve.'

    # Llamamos al LLM. Devuelve un AIMessage.
    response = llm.invoke(prompt)

    # Escribimos solo el campo nuevo.
    return {'message': response.content}

def format_node(state: GreetState) -> dict:
    # Este nodo consume lo que produjo greet_node.
    message = state['message']

    # Produce un campo nuevo para el resultado final.
    return {'formatted': f'>> {message} <<'}


## Construccion del grafo

Registrar un nodo no lo ejecuta. Solo le dice al grafo que esa funcion existe.

Los edges definen el orden:

```txt
START -> greet_node -> format_node -> END
```


In [ ]:
builder = StateGraph(GreetState)

# Asociamos nombres del grafo con funciones Python.
builder.add_node('greet_node', greet_node)
builder.add_node('format_node', format_node)

# Declaramos el flujo.
builder.add_edge(START, 'greet_node')
builder.add_edge('greet_node', 'format_node')
builder.add_edge('format_node', END)

# compile valida que el grafo tenga entrada, salida y nodos conectados.
graph = builder.compile()


result = graph.invoke({'name': 'Ada'})

print(result)
print('Respuesta final:', result['formatted'])
